# Exp 27 — Octopus clip extractor (motion + >50% visible)

For each unprocessed Right-camera video on the remote repo:
1. Stream **1 frame/sec** (single ffmpeg pass) and compute, per second:
   - `p_visible` from the CLIP+MLP model (`weights/clip_mlp_best.pt`)
   - `motion` via grayscale frame-differencing (normalised per video)
2. Slide a **20 s** window; keep non-overlapping windows where
   **>50 % of frames are octopus-visible** *and* mean motion ≥ threshold.
3. Extract each qualifying 20 s window locally (ffmpeg segment copy — only the
   clip's bytes are fetched).
4. Record the clip (video URL + time location) in `data/octopus_clips_index.json`
   and mark the video done in `data/clip_pipeline_processed.json` so it is never reprocessed.


In [ ]:
import json, subprocess, re, urllib.parse, datetime, time
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import torch, torch.nn as nn
from PIL import Image

# project root = nearest parent containing data/ and weights/
PROJECT = next(p for p in [Path.cwd(), *Path.cwd().parents]
               if (p/"data").exists() and (p/"weights").exists())
print("PROJECT:", PROJECT)

CKPT_PATH   = PROJECT/"weights"/"clip_mlp_best.pt"
CLIPS_DIR   = PROJECT/"data"/"octopus_clips_auto"
CLIPS_INDEX = PROJECT/"data"/"octopus_clips_index.json"
PROCESSED   = PROJECT/"data"/"clip_pipeline_processed.json"

BASE      = "https://repo.octopus-intelligence.org/public/O-vulgaris-Nity-2026-2-20--"
import sys as _sys
from pathlib import Path as _P
_root = next(p for p in [_P.cwd(), *_P.cwd().parents] if (p / 'server_creds.py').exists())
_sys.path.insert(0, str(_root))   # repo root, wherever this nb is run from
from server_creds import USER, PASS as PWD  # creds from env / .env, not hardcoded

# ---- pipeline params (tune here) ----
CAMERAS          = ["Right Back", "Right Front", "Right Left",
                    "Right Right", "Right Top"]   # all Right cameras
SAMPLE_FPS       = 1.0               # 1 frame/sec for motion + visibility
CLIP_LEN         = 20                # seconds per clip
MIN_VISIBLE_FRAC = 0.5               # > this fraction of frames must show the octopus
VIS_THRESH       = 0.5               # p_visible >= this -> frame counts as "visible"
MOTION_THRESH    = 0.12              # mean normalised motion within the window
SIZE, BATCH      = 224, 64
LIMIT            = None               # max videos to process this run (None = all)

CLIPS_DIR.mkdir(parents=True, exist_ok=True)
device = ("mps" if torch.backends.mps.is_available()
          else "cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

In [ ]:
# ---- server enumeration ----
def _curl(url):
    return subprocess.run(["curl","-s","--user",f"{USER}:{PWD}",url],
                          capture_output=True, text=True).stdout

def list_dates():
    out=_curl(f"{BASE}/Right%20Top/Local/")
    return sorted(set(re.findall(r'href="(\d{4}-\d{2}-\d{2})/"', out)))

def list_segments(cam, date):
    enc=urllib.parse.quote(cam); out=_curl(f"{BASE}/{enc}/Local/{date}/")
    rows=[]
    for f in re.findall(r'href="([^"]+\.mp4)"', out):
        m=re.match(r"(\d+)--", f)
        if not m: continue
        seg=m.group(1); cam_us=cam.replace(" ","_")
        rows.append({"video":f"data/aquarium/full/{date}/{seg}/{cam_us}.mp4",
                     "date":date, "segment":seg, "camera":cam_us,
                     "url":f"{BASE}/{enc}/Local/{date}/{f}"})
    return rows

def enumerate_candidates(dates, cams):
    tasks=[(c,d) for d in dates for c in cams]; out=[]
    with ThreadPoolExecutor(max_workers=16) as ex:
        for r in ex.map(lambda a: list_segments(*a), tasks): out.extend(r)
    return out

In [ ]:
# ---- registries ----
def load_json(path, default):
    return json.load(open(path)) if path.exists() else default

def init_registries():
    proc = load_json(PROCESSED, {
        "task":"octopus_clip_extraction",
        "description":"Videos processed by the 20s-clip pipeline (motion + >50% octopus visible). Do not reprocess.",
        "updated_at":None, "count":0, "processed":[]})
    idx = load_json(CLIPS_INDEX, {
        "description":"Extracted 20s octopus clips: source video url + time location.",
        "updated_at":None, "count":0, "clips":[]})
    return proc, idx

def save_registries(proc, idx):
    now=datetime.datetime.now().isoformat(timespec="seconds")
    proc["count"]=len(proc["processed"]); proc["updated_at"]=now
    idx["count"]=len(idx["clips"]);        idx["updated_at"]=now
    json.dump(proc, open(PROCESSED,"w"), indent=2)
    json.dump(idx,  open(CLIPS_INDEX,"w"), indent=2)

In [ ]:
# ---- model ----
def load_model():
    try:
        import pkg_resources, packaging, packaging.version, packaging.specifiers, packaging.requirements
        pkg_resources.packaging=packaging
    except Exception:
        pass
    import clip as clip_lib
    ckpt=torch.load(CKPT_PATH, map_location=device)
    cm,pp=clip_lib.load(ckpt["clip_model"], device=device); cm.eval()
    feat=ckpt["feat_dim"]; arch=ckpt.get("arch","linear")
    hid=[int(x) for x in arch.replace("mlp_","").split("_")]; dims=[feat]+hid+[2]
    layers=[]
    for i in range(len(dims)-1):
        layers.append(nn.Linear(dims[i],dims[i+1]))
        if i<len(dims)-2: layers+=[nn.ReLU(), nn.Dropout(0.3)]
    clf=nn.Sequential(*layers).to(device); clf.load_state_dict(ckpt["state_dict"]); clf.eval()
    vis=ckpt.get("label_map",{}).get("visible",1)
    print(f"model: {ckpt['clip_model']}+{arch}  acc={ckpt.get('test_acc',0):.1%}")
    return cm,pp,clf,vis

clip_model, preprocess, classifier, vis_idx = load_model()

In [ ]:
# ---- single 1fps stream -> per-second p_visible + motion ----
def scan_video(url):
    auth=url.replace("https://", f"https://{USER}:{PWD}@")
    cmd=["ffmpeg","-loglevel","error","-i",auth,
         "-vf",(f"fps={SAMPLE_FPS},scale={SIZE}:{SIZE}:force_original_aspect_ratio=decrease,"
                f"pad={SIZE}:{SIZE}:-1:-1:color=gray"),  # letterbox (no crop) — matches training
         "-f","image2pipe","-vcodec","rawvideo","-pix_fmt","rgb24","-"]
    proc=subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    fsize=SIZE*SIZE*3
    pv=[]; motion=[]; prev=None; buf=[]
    def flush():
        if not buf: return
        batch=torch.stack([preprocess(im) for im in buf]).to(device)
        with torch.no_grad():
            f=clip_model.encode_image(batch).float(); f=f/f.norm(dim=-1,keepdim=True)
            p=torch.softmax(classifier(f),dim=1)[:,vis_idx]
        pv.extend(p.cpu().tolist()); buf.clear()
    while True:
        raw=proc.stdout.read(fsize)
        if len(raw)<fsize: break
        arr=np.frombuffer(raw,np.uint8).reshape(SIZE,SIZE,3)
        gray=arr.mean(axis=2)
        motion.append(0.0 if prev is None else float(np.mean(np.abs(gray-prev))))
        prev=gray
        buf.append(Image.fromarray(arr))
        if len(buf)>=BATCH: flush()
    flush()
    proc.stdout.close(); proc.wait()
    pv=np.array(pv,np.float32); motion=np.array(motion,np.float32)
    if motion.max()>0: motion=motion/motion.max()   # normalise per video
    return pv, motion

In [ ]:
# ---- find non-overlapping 20s windows (visible + motion) ----
def find_clip_windows(pv, motion):
    L=int(CLIP_LEN*SAMPLE_FPS); N=len(pv); out=[]; s=0
    while s+L<=N:
        win_pv=pv[s:s+L]; win_mo=motion[s:s+L]
        vis_frac=float((win_pv>=VIS_THRESH).mean())
        mean_mo =float(win_mo.mean())
        if vis_frac>MIN_VISIBLE_FRAC and mean_mo>=MOTION_THRESH:
            out.append({"start_sec":int(s/SAMPLE_FPS), "end_sec":int((s+L)/SAMPLE_FPS),
                        "visible_frac":round(vis_frac,3), "mean_motion":round(mean_mo,3)})
            s+=L            # non-overlapping
        else:
            s+=1
    return out

def extract_clip(url, start, end, out_path):
    out_path.parent.mkdir(parents=True, exist_ok=True)
    if out_path.exists() and out_path.stat().st_size>10000: return True
    auth=url.replace("https://", f"https://{USER}:{PWD}@")
    cmd=["ffmpeg","-loglevel","error","-y","-ss",str(start),"-to",str(end),
         "-i",auth,"-c:v","copy","-c:a","aac",str(out_path)]
    r=subprocess.run(cmd, capture_output=True, text=True)
    return r.returncode==0 and out_path.exists()

In [ ]:
# ---- main loop ----
proc_reg, clip_idx = init_registries()
done = {r["video"] for r in proc_reg["processed"]}

dates = list_dates()
cands = enumerate_candidates(dates, CAMERAS)
todo  = [c for c in cands if c["video"] not in done]
if LIMIT: todo = todo[:LIMIT]
print(f"{len(cands)} candidate videos; {len(todo)} to process (cameras={CAMERAS})")

for i,c in enumerate(todo,1):
    t0=time.perf_counter()
    print(f"[{i}/{len(todo)}] {c['date']} {c['segment']} {c['camera']}", flush=True)
    try:
        pv, motion = scan_video(c["url"])
    except Exception as e:
        print("  ! scan failed:", e); continue
    if len(pv)==0:
        print("  ! no frames"); continue
    windows = find_clip_windows(pv, motion)
    n_saved=0
    for w in windows:
        clip_path = CLIPS_DIR/c["date"]/c["segment"]/f"{c['camera']}_{w['start_sec']:04d}-{w['end_sec']:04d}.mp4"
        if extract_clip(c["url"], w["start_sec"], w["end_sec"], clip_path):
            clip_idx["clips"].append({
                "video":c["video"], "video_url":c["url"], "date":c["date"],
                "segment":c["segment"], "camera":c["camera"],
                "start_sec":w["start_sec"], "end_sec":w["end_sec"],
                "visible_frac":w["visible_frac"], "mean_motion":w["mean_motion"],
                "clip_path":str(clip_path.relative_to(PROJECT)),
                "added_at":datetime.datetime.now().isoformat(timespec="seconds")})
            n_saved+=1
    proc_reg["processed"].append({"video":c["video"], "date":c["date"],
        "segment":c["segment"], "camera":c["camera"],
        "n_clips":n_saved, "n_frames":int(len(pv)), "sources":["exp27_clip_pipeline"]})
    done.add(c["video"]); save_registries(proc_reg, clip_idx)
    print(f"   {len(pv)} frames | {len(windows)} windows -> {n_saved} clips | "
          f"{time.perf_counter()-t0:.0f}s", flush=True)

print("DONE. clips:", clip_idx["count"], "| processed videos:", proc_reg["count"])

In [ ]:
# ---- results ----
import pandas as pd
idx=json.load(open(CLIPS_INDEX))
print("total clips:", idx["count"])
df=pd.DataFrame(idx["clips"])
df if len(df) else "no clips yet — try lowering MOTION_THRESH or widening CAMERAS"